# AdaMask Ablation Test: dataset x adaptive-loss

A self-contained test harness for proving the AdaMask idea works on something conversational, without touching anything under `adamask/`. Reuses the model, diffusion schedule, and sampler unchanged; only two things are implemented locally in this notebook instead:

1. **Dialog-formatted data loading** for `daily_dialog` (a real conversational dataset), since `adamask/data.py`'s windowing is built for raw prose and would chop a dialogue mid-turn at arbitrary token boundaries.
2. **An on/off switch for the adaptive difficulty-weighted loss**, so a `daily_dialog` run (or a `tinystories` run) can be compared directly with and without the AdaMask idea, all else equal.

Edit the `PARAMS` cell below and rerun to change what gets tested. Nothing here modifies the real package -- everything is imported read-only from `adamask`.

Before running: **Runtime > Change runtime type > GPU** (if on Colab).

In [ ]:
import os

# Local: this notebook lives inside experiments/adaptive_loss_ablation/ in an
# existing checkout of this repo. Checking a known absolute path (rather than
# a relative one like "../../adamask") doesn't depend on whatever directory
# the kernel happens to start in -- a relative check silently failed and
# re-cloned a nested copy anyway when the kernel's cwd wasn't what it assumed.
# Colab: that path won't exist there, so it falls through to a fresh clone.
_LOCAL_REPO = "/home/lenue/Desktop/AdaMask/AdaMask322"

if os.path.exists("adamask"):
    print("already at repo root:", os.getcwd())
elif os.path.exists(os.path.join(_LOCAL_REPO, "adamask")):
    os.chdir(_LOCAL_REPO)
    print("moved to repo root:", os.getcwd())
else:
    !git clone https://github.com/Lenue01/AdaMask322.git
    %cd AdaMask322
    !pip install -q transformers datasets tqdm

## Parameters -- edit these and rerun

- `DATASET`: `"tinystories"` (simple narrative prose, already proven to work) or `"daily_dialog"` (real conversational turns -- the actual target)
- `USE_ADAPTIVE_LOSS`: `True` for the AdaMask idea (loss weighted by per-token difficulty), `False` for a plain-uniform-loss baseline to compare against

Architecture/training knobs default to the same small, Colab-friendly scale already validated in `sanity_check.py`.

In [ ]:
import torch

# ---- What to test ----
DATASET = "daily_dialog"      # "tinystories" or "daily_dialog"
USE_ADAPTIVE_LOSS = True      # True = AdaMask's difficulty-weighted loss, False = plain uniform-loss baseline

# ---- Architecture ----
CONTEXT_LENGTH = 128
HIDDEN_SIZE = 256
HEADS = 8
LAYERS = 8
STEPS = 32                    # diffusion timesteps / masking levels

# ---- Training ----
BATCH_SIZE = 32
NUM_EPOCHS = 10
STEPS_PER_EPOCH = 1500
WARMUP_STEPS = 100
SAVE_EVERY_EPOCHS = 1
SAVE_CHECKPOINTS = True       # False skips writing any .pt files -- useful for quick throwaway runs
ACCUM_STEPS = 1
DIFFICULTY_LOSS_SCALE = 0.3   # weight = 1 + scale * difficulty; only applied when USE_ADAPTIVE_LOSS is True
DIFFICULTY_DECAY = 0.999      # EMA decay for per-token difficulty stats
LR = None                     # None = auto-scale from HIDDEN_SIZE/LAYERS (see adamask/config.py)

# ---- Validation ----
VAL_BATCHES = 16
VAL_SEED = 1234

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## Dataset setup

`tinystories` reuses `adamask.data.get_dataloader` completely unchanged. `daily_dialog` gets formatted into the same `text`-shaped rows `adamask.data._tokenize_batch` already expects (one row per dialogue, turns joined with alternating `A:`/`B:` labels), then reuses that exact same (already-tested) windowing/tokenization function -- nothing about the real package is modified, this just feeds it turn-structured data instead of raw prose.

In [ ]:
import dataclasses
from datasets import load_dataset
from torch.utils.data import DataLoader

from adamask.config import Config
from adamask.data import get_dataloader, _tokenize_batch

DATASET_SETTINGS = {
    "tinystories": dict(
        dataset_name="roneneldan/TinyStories", dataset_config="default",
        split="train", val_split="validation", format="text",
    ),
    "daily_dialog": dict(
        dataset_name="OpenRL/daily_dialog", dataset_config=None,
        split="train", val_split="validation", format="dialog",
    ),
}
settings = DATASET_SETTINGS[DATASET]

config = Config(
    dataset_name=settings["dataset_name"], dataset_config=settings["dataset_config"], split=settings["split"],
    context_length=CONTEXT_LENGTH, hidden_size=HIDDEN_SIZE, heads=HEADS, layers=LAYERS, steps=STEPS,
    batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS, steps_per_epoch=STEPS_PER_EPOCH, warmup_steps=WARMUP_STEPS,
    save_every_epochs=SAVE_EVERY_EPOCHS, accum_steps=ACCUM_STEPS,
    difficulty_loss_scale=DIFFICULTY_LOSS_SCALE, difficulty_decay=DIFFICULTY_DECAY,
    val_split=settings["val_split"], val_batches=VAL_BATCHES, val_seed=VAL_SEED, lr=LR,
)
config.device = DEVICE
dataset_format = settings["format"]


def _format_dialog_batch(examples):
    """One row per dialogue, turns joined with alternating speaker labels --
    never windowed/concatenated across separate dialogues the way raw prose is."""
    texts = []
    for turns in examples["dialog"]:
        lines = [f"{'A' if i % 2 == 0 else 'B'}: {t.strip()}" for i, t in enumerate(turns)]
        texts.append("\n".join(lines))
    return {"text": texts}


def build_dataloader(config, split, dataset_format):
    if dataset_format == "text":
        return get_dataloader(dataclasses.replace(config, split=split))

    dataset = load_dataset(config.dataset_name, split=split, streaming=True)
    dataset = dataset.shuffle(seed=42, buffer_size=10_000)
    dataset = dataset.map(_format_dialog_batch, batched=True, remove_columns=dataset.column_names)
    # dataset.column_names is None here (streaming datasets can't infer schema
    # after an arbitrary .map() without running it) -- remove_columns must name
    # _format_dialog_batch's own known output column explicitly, or the stale
    # "text" column survives at the wrong row count and .map() raises a length
    # mismatch against the windowed input_ids/attention_mask it just produced.
    dataset = dataset.map(
        lambda batch: _tokenize_batch(batch, config.tokenizer, config),
        batched=True, remove_columns=["text"],
    )
    dataset = dataset.with_format("torch")
    return DataLoader(dataset, batch_size=config.batch_size, num_workers=config.max_workers, pin_memory=True)


dataloader = build_dataloader(config, config.split, dataset_format)

# Sanity-check the formatting before spending any training time on it.
batch = next(iter(dataloader))
print(f"dataset: {DATASET} ({dataset_format} format), batch shape: {tuple(batch['input_ids'].shape)}")
print("first example decoded:")
print(config.tokenizer.decode(batch["input_ids"][0].tolist(), skip_special_tokens=True))

## Model + diffusion

Unchanged from the real package.

In [ ]:
from adamask.model import MaskedDiffusionTransformer
from adamask.diffusion import TokenDifficulty

model = MaskedDiffusionTransformer(config).to(config.device)
diffusion = TokenDifficulty(
    config.vocab_size, config.mask_token_id, config.pad_token_id, config.steps, config.device,
    decay=config.difficulty_decay,
)
print(f"params: {sum(p.numel() for p in model.parameters()):,}")

## Ablation training loop

Same loop as `adamask.train.train`, reusing its helper functions (`get_lr`, `compute_val_loss`, `print_epoch_samples`, `save_checkpoint`, `load_checkpoint`, `print_token_stats`) unchanged. Two differences, both scoped to this experiment:

- `use_adaptive_loss` switches the loss weight between `1 + scale * difficulty` (AdaMask) and a flat `1.0` (baseline) -- difficulty stats are tracked identically either way, so both arms stay directly comparable; only whether the stats get *used* to weight the loss differs.
- Validation batches are built with the `build_dataloader` wrapper above (so `daily_dialog` works), instead of `adamask.train.build_val_batches`, which only understands `text`-format datasets.

In [ ]:
import torch.nn.functional as F
from tqdm import tqdm
from adamask.train import get_lr, compute_val_loss, print_epoch_samples, save_checkpoint, load_checkpoint, print_token_stats


def build_val_batches_local(config, dataset_format):
    try:
        val_loader = build_dataloader(config, config.val_split, dataset_format)
        val_iter = iter(val_loader)
    except ValueError as e:
        print(f"WARNING: couldn't load validation split {config.val_split!r} ({e}); continuing without validation.")
        return []
    batches = []
    for _ in range(config.val_batches):
        try:
            batch = next(val_iter)
        except StopIteration:
            break
        tokens = batch["input_ids"].to(config.device)
        pad_mask = batch["attention_mask"].to(config.device) == 0
        batches.append((tokens, pad_mask))
    return batches


def train_ablation(model, diffusion, dataloader, config, dataset_format, use_adaptive_loss, tag, save_checkpoints=True, resume_path=None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0, weight_decay=config.weight_decay)
    use_amp = config.device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    data_iter = iter(dataloader)
    val_batches = build_val_batches_local(config, dataset_format)

    start_epoch = 0
    if resume_path:
        start_epoch = load_checkpoint(resume_path, model, optimizer, scaler, diffusion, config)
        print(f"Resumed from {resume_path}, continuing at epoch {start_epoch}")

    for epoch in range(start_epoch, config.num_epochs):
        model.train()
        running_loss = 0.0
        loop = tqdm(range(config.steps_per_epoch), desc=f"[{tag}] Epoch {epoch}")
        for step in loop:
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(dataloader)
                batch = next(data_iter)

            tokens = batch["input_ids"].to(config.device)
            attn_msk = batch["attention_mask"].to(config.device)
            pad_mask = attn_msk == 0

            global_step = epoch * config.steps_per_epoch + step
            lr = get_lr(global_step, config.warmup_steps, config.lr, config.lr / 10, config.total_steps)
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr

            if step % config.accum_steps == 0:
                optimizer.zero_grad(set_to_none=True)
                did_backward_this_cycle = False

            t = torch.randint(1, config.steps + 1, (tokens.size(0),), device=config.device)
            x_t, is_masked = diffusion.corrupt(tokens, t)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(x_t, t, key_padding_mask=pad_mask)
                loss_mask = (is_masked & ~pad_mask).bool()
                if loss_mask.any():
                    target = tokens[loss_mask]
                    difficulty = diffusion.get_difficulty(target)
                    if use_adaptive_loss:
                        weight = 1.0 + config.difficulty_loss_scale * difficulty
                    else:
                        weight = torch.ones_like(difficulty)
                    per_token_loss = F.cross_entropy(
                        logits[loss_mask], target, label_smoothing=0.05, reduction="none",
                    )
                    loss = (per_token_loss * weight).mean() / config.accum_steps
                else:
                    loss = None

            if loss is not None:
                scaler.scale(loss).backward()
                did_backward_this_cycle = True

            # At very fine diffusion schedules (large STEPS) combined with a small
            # batch, a batch can by chance draw only low-t (near-zero mask rate)
            # examples and end up with literally zero masked tokens -- loss is
            # None for the whole step, and calling scaler.step() with no backward()
            # having been recorded this cycle raises "No inf checks were recorded
            # for this optimizer." Only step if backward() actually ran.
            if (step + 1) % config.accum_steps == 0 and did_backward_this_cycle:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            diffusion.update(logits.detach(), tokens, is_masked)

            if loss is not None:
                running_loss += loss.item() * config.accum_steps
            loop.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

        if save_checkpoints and (epoch % config.save_every_epochs == 0 or epoch == config.num_epochs - 1):
            save_checkpoint(model, optimizer, scaler, diffusion, epoch, config, f"{tag}_epoch_{epoch}.pt")

        if epoch % config.save_every_epochs == 0 or epoch == config.num_epochs - 1:
            print(f"\n--- [{tag}] Epoch {epoch} Token Difficulty ---")
            print_token_stats(diffusion, config.tokenizer)

        val_loss = compute_val_loss(model, diffusion, val_batches, config)
        if val_loss is not None:
            print(f"  [{tag}] Val loss (fixed mask seed): {val_loss:.4f}")

        print_epoch_samples(model, diffusion, config, epoch)
        print()

## Run

Checkpoints save as `{dataset}_{adaptive|uniform}_epoch_N.pt`, so different `DATASET`/`USE_ADAPTIVE_LOSS` combinations from the PARAMS cell above never collide -- rerun with a different combination and compare the printed samples/val-loss side by side.

In [ ]:
tag = f"{DATASET}_{'adaptive' if USE_ADAPTIVE_LOSS else 'uniform'}"
print(f"Training: {tag}")
train_ablation(model, diffusion, dataloader, config, dataset_format, USE_ADAPTIVE_LOSS, tag, save_checkpoints=SAVE_CHECKPOINTS)

## Generate a few more samples

Uses `adamask.sample.sample` unchanged -- fully dataset-agnostic.

In [ ]:
from adamask.sample import sample

tokens = sample(model, diffusion, config, num_samples=4, temperature=0.9)
for i, row in enumerate(tokens.tolist()):
    text = config.tokenizer.decode(row, skip_special_tokens=True)
    print(f"--- sample {i} ---")
    print(text)
    print()